This is a test nb for LightRAG with pdf and azure

In [16]:
import pdfplumber

pdf_path = "../data/Skedsmo VGS konkurransegrunnlag del I.pdf"  # Constitution_of_India.pdf
pdf_text = ""
text_path = "../data/Anbud1.txt"

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        pdf_text += page.extract_text() + "\n"
#print(pdf_text)

f = open(text_path,"w", encoding='utf-8')
f.write(pdf_text)
f.close()



In [1]:
import os
import asyncio
from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc
import numpy as np
from dotenv import load_dotenv
import logging
from openai import AzureOpenAI

import nest_asyncio
nest_asyncio.apply()

logging.basicConfig(level=logging.INFO)

load_dotenv()

AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")

AZURE_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_EMBEDDING_DEPLOYMENT")
AZURE_EMBEDDING_API_VERSION = os.getenv("AZURE_EMBEDDING_API_VERSION")

WORKING_DIR = "../data RAG/Nye Rikshospitalet"

#if os.path.exists(WORKING_DIR):
#    import shutil
#
#    print("Removing existing working directory")
#    shutil.rmtree(WORKING_DIR)
#
#os.mkdir(WORKING_DIR)


async def llm_model_func(
    prompt, system_prompt=None, history_messages=[], keyword_extraction=False, **kwargs
) -> str:
    client = AzureOpenAI(
        api_key=AZURE_OPENAI_API_KEY,
        api_version=AZURE_OPENAI_API_VERSION,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
    )

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    if history_messages:
        messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})

    chat_completion = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,  # model = "deployment_name".
        messages=messages,
        temperature=kwargs.get("temperature", 0),
        top_p=kwargs.get("top_p", 1),
        n=kwargs.get("n", 1),
    )
    return chat_completion.choices[0].message.content


async def embedding_func(texts: list[str]) -> np.ndarray:
    client = AzureOpenAI(
        api_key=AZURE_OPENAI_API_KEY,
        api_version=AZURE_EMBEDDING_API_VERSION,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
    )
    embedding = client.embeddings.create(model=AZURE_EMBEDDING_DEPLOYMENT, input=texts)

    embeddings = [item.embedding for item in embedding.data]
    return np.array(embeddings)


#async def test_funcs():
#    result = await llm_model_func("How are you?")
#    print("Resposta do llm_model_func: ", result)
#
#    result = await embedding_func(["How are you?"])
#    print("Resultado do embedding_func: ", result.shape)
#    print("Dimensão da embedding: ", result.shape[1])


#asyncio.run(test_funcs())

embedding_dimension = 3072

rag = LightRAG(
    working_dir=WORKING_DIR,
    llm_model_func=llm_model_func,
    embedding_func=EmbeddingFunc(
        embedding_dim=embedding_dimension,
        max_token_size=8192,
        func=embedding_func,
    ),
)

#book1 = open("./examples/datatest/Anbud1.txt", encoding="utf-8")
#book2 = open("./examples/datatest/Anbud2.txt", encoding="utf-8")

#rag.insert([book1.read(), book2.read()])

query_text = "What are the main themes?"

#print("Result (Naive):")
#print(rag.query(query_text, param=QueryParam(mode="naive")))

#print("\nResult (Local):")
#print(rag.query(query_text, param=QueryParam(mode="local")))
#
print("\nResult (Global):")
print(rag.query(query_text, param=QueryParam(mode="global")))
#
#print("\nResult (Hybrid):")
#print(rag.query(query_text, param=QueryParam(mode="hybrid")))


c:\Users\erlmelby\OneDrive - Veidekke\04 Utvikler\Artificial intelligence\LightRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:lightrag:Logger initialized for working directory: ../data RAG/Nye Rikshospitalet
INFO:lightrag:Load KV llm_response_cache with 4 data
INFO:lightrag:Load KV full_docs with 1 data
INFO:lightrag:Load KV text_chunks with 1110 data
INFO:lightrag:Loaded graph from ../data RAG/Nye Rikshospitalet\graph_chunk_entity_relation.graphml with 10883 nodes, 13064 edges
INFO:nano-vectordb:Load (10845, 3072) data
INFO:nano-vectordb:Init {'embedding_dim': 3072, 'metric': 'cosine', 'storage_file': '../data RAG/Nye Rikshospitalet\\vdb_entities.json'} 10845 data
INFO:nano-vectordb:Load (13064, 3072) data
INFO:nano-vectordb:Init {'embedding_dim': 3072, 'metric': 'cosine', 'storage_file': '../data


Result (Global):
The main themes of the Nye Rikshospitalet project can be categorized into several key areas, each reflecting the comprehensive and multifaceted nature of this significant construction and development initiative. Below, I will outline these themes in detail:

### 1. **Construction and Development**
Nye Rikshospitalet is a large-scale construction project aimed at establishing a new hospital in Gaustad, Oslo, Norway. The project involves extensive construction activities, including the development of new buildings, infrastructure, and various systems such as water and sewage management. The construction period began in autumn 2023 and is expected to last until 2031. The project is managed by a dedicated project organization led by a project director and involves multiple contractors and enterprises.

### 2. **Healthcare Services**
The project focuses on consolidating regional and national specialized functions into a complete regional hospital. It aims to provide local 

#Init rag

In [22]:
prequery = "Svar på norsk"
query = "hvilke krav gjelder for klima og miljø?"
print(f"{prequery} {query}")


Svar på norsk Svar på norsk. Hvem er oppdragsgiveren og hvor mange ansatte har de?


In [3]:
from lightrag import QueryParam

prequery = "Svar på norsk."
#query = "Hvem er oppdragsgiveren og hvor mange ansatte har de?"
#query = "Si litt om tomta og området rundt."
#query = "Hva sies det om opsjoner?"
query = "hvilke krav gjelder for klima og miljø?"
result = rag.query(prequery + " " + query, param=QueryParam(mode="hybrid"))  # Choose the appropriate mode
print(prequery + " " + query + "\n")
print(result)


INFO:httpx:HTTP Request: POST https://swedencentral.api.cognitive.microsoft.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-08-01-preview "HTTP/1.1 200 OK"
INFO:lightrag:kw_prompt result:
INFO:lightrag:Using hybrid mode for query processing


{
  "high_level_keywords": ["Klima", "Miljø", "Krav"],
  "low_level_keywords": ["Norsk", "Reguleringer", "Miljøstandarder", "Klimaavtaler"]
}


INFO:httpx:HTTP Request: POST https://swedencentral.api.cognitive.microsoft.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2023-05-15 "HTTP/1.1 200 OK"
INFO:lightrag:Local query uses 60 entites, 115 relations, 3 text units
INFO:httpx:HTTP Request: POST https://swedencentral.api.cognitive.microsoft.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2023-05-15 "HTTP/1.1 200 OK"
INFO:lightrag:Global query uses 26 entites, 60 relations, 3 text units
INFO:httpx:HTTP Request: POST https://swedencentral.api.cognitive.microsoft.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-08-01-preview "HTTP/1.1 200 OK"


Svar på norsk. hvilke krav gjelder for klima og miljø?

### Krav til Klima og Miljø i Nye Rikshospitalet Prosjektet

Nye Rikshospitalet-prosjektet har omfattende krav til klima og miljø som er nøye dokumentert og implementert gjennom ulike retningslinjer og standarder. Disse kravene er utformet for å sikre at prosjektet oppfyller høye miljøstandarder og bidrar til bærekraftig utvikling. Her er en oversikt over de viktigste kravene:

#### 1. **Bilag D8: Krav til Klima og Miljø i Byggefase**
Bilag D8 er et sentralt dokument som spesifiserer generelle krav til klima og miljø under byggefasen av Nye Rikshospitalet. Dette dokumentet dekker en rekke aspekter, inkludert miljøoppfølgingsplaner, avfallsreduksjon, og tiltak for å sikre et rent og tørt bygg. Det er også referanser til BREEAM-NOR 2016 manualen, som gir detaljerte kriterier og tiltak som skal følges.

#### 2. **Miljøoppfølgingsplan (MOP)**
Miljøoppfølgingsplanen beskriver prosjektets miljømål og tiltak som entreprenøren skal forhol